# Data preprocessing

In order to develop a proper model for the estimation of the surface water fracture, we first must adapt the data of previous satelites so that they can be used in our model.

The datasets that we will use are:

* Ku Band (GCOM-W1 satelite): https://nuvol.uv.es/owncloud/index.php/s/aIFQieObF2FMRKK
* Ka Band (GCOM-W1 satelite): https://nuvol.uv.es/owncloud/index.php/s/R3DpqwDSgxoZdPl
* LPDR: http://files.ntsg.umt.edu/data/LPDR_v3/GeoTif/2017/

### Library loading

In [1]:
import xarray as xr
import numpy as np
import os
import zipfile
import io
import re
import requests

from datetime import datetime, date, timedelta

time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)

In [2]:
from osgeo import gdal, osr

ModuleNotFoundError: No module named 'osgeo'

## Data download

We'll start by downloading all of the satellite data.

In [ ]:
BASE_URL = (
    "https://data.remss.com/TB/intercalibration/"
    "windsat_TB_maps_daily_025deg_unfiltered/"
)

OUTPUT_DIR = os.path.join("data", "WINDSAT")
os.makedirs(OUTPUT_DIR, exist_ok=True)

start_date = date(2017, 1, 1)
end_date = date(2017, 12, 31)

current_date = start_date

while current_date <= end_date:
    datestr = current_date.strftime("%Y_%m_%d")
    filename = f"RSS_WINDSAT_DAILY_TBTOA_MAPS_{datestr}.nc"
    url = BASE_URL + filename
    local_path = os.path.join(OUTPUT_DIR, filename)

    if os.path.exists(local_path):
        print(f"Skipping (already exists): {filename}")
        current_date += timedelta(days=1)
        continue

    print(f"Downloading: {filename}")

    try:
        response = requests.get(url, stream=True, timeout=60)
        response.raise_for_status()

        with open(local_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

    except requests.RequestException as e:
        print(f"Failed to download {filename}: {e}")

    current_date += timedelta(days=1)

print("Download process completed.")

Downloaded RSS_WINDSAT_DAILY_TBTOA_MAPS_2017_01_01.nc


## Data analysis

Now that we have all of our data, let's check it's structure and see if it's properly formated.

### LPDR

In [2]:
lpdr_ds = xr.open_dataset("data/LPDR/2017tif/AMSRU_Mland_2017001A.tif")

print("Dimensions:", dict(lpdr_ds.sizes))
print("Coordinates:")
for coord in ["x", "y"]:
    if coord in lpdr_ds.coords:
        print(f"  {coord}: {lpdr_ds.coords[coord].values[0]}  →  {lpdr_ds.coords[coord].values[-1]}")

Dimensions: {'band': 7, 'x': 1383, 'y': 586}
Coordinates:
  x: -17321659.775000002  →  17321659.77499638
  y: 7332251.062494965  →  -7332251.0625035055


The LPDR dataset has an EASE-v2 projection.

## Windsat bands

In [4]:
windsat_ds = xr.open_dataset("data/WINDSAT/RSS_WINDSAT_DAILY_TBTOA_MAPS_2017_01_01.nc", decode_times=time_coder)
windsat_ds = windsat_ds.set_coords(["latitude", "longitude"])

print("Dimensions:", dict(windsat_ds.sizes))
print("Coordinates:")
for coord in ["latitude", "longitude"]:
    if coord in windsat_ds.coords:
        print(f"  {coord}: {windsat_ds.coords[coord].values[0]}  →  {windsat_ds.coords[coord].values[-1]}")

Dimensions: {'longitude_grid': 1440, 'latitude_grid': 720, 'swath_sector': 2, 'look_direction': 2, 'frequency_band': 5, 'polarization': 4, 'polarization_dual': 2}
Coordinates:
  latitude: -89.875  →  89.875
  longitude: 0.125  →  359.875


The windsat dataset has a simple EQR projection.

## Data projection

Since both datasets do not share the same grid, we need to reproject them to a common grid. We'll do this by reprojecting the LPDR dataset to the same projection as the Windsat dataset.

PROBLEMA: LA VARIABLE LATITUD NO SIGUE UN ESPACIADO CONSTANTE DE 0.25

In [ ]:
def reproject_file(file_path: str, output_folder: str = None) -> bool:
    """
        Read the input geotiff in EASE v1
        Reproject + resample into ED 0.25º
        create latitude an longitude bands for convenience.
        (bands added as second to last, and last band respectively)

        param output_folder: name of the new file to save reprojected data.
            If None, data will be ovewritten in file_path.

        Returns bool: whether or not the file was succesfully reprojected
    """
    dataset = gdal.Open(file_path)

    # Define the geotransform for the output dataset
    target_geotransform = (-180.0, 0.25, 0.0, 90.0, 0.0, -0.25)

    output_width = 1388
    output_height = 584

    # Define src and geotransform from the input:
    source_srs = osr.SpatialReference()
    source_srs.ImportFromEPSG(3410)

    # Desired projection
    target_srs = osr.SpatialReference()
    target_srs.ImportFromEPSG(4326)

    # Declare the output file and driver:
    driver = gdal.GetDriverByName("GTiff")

    if output_folder is None:
        output_file = os.path.join(os.path.dirname(file_path), "temp.tif")

    else:
        output_file = os.path.join(output_folder, os.path.basename(file_path))

    # Create the new dataset
    output_dataset = driver.Create(
        output_file, output_width, output_height,
        dataset.RasterCount,
        gdal.GDT_Float32
    )
    output_dataset.SetProjection(target_srs.ExportToWkt())
    output_dataset.SetGeoTransform(target_geotransform)

    # Set the output dataset value to -999.0, instead of 0.
    for i in range(1, output_dataset.RasterCount, 1):
        band = output_dataset.GetRasterBand(i)
        arr = band.ReadAsArray()
        arr = arr - 999.0
        band.WriteArray(arr)

    # Reproject and resample using gdal.Warp()
    gdal.Warp(
        output_dataset,
        dataset,
        dstSRS=target_srs.ExportToWkt(),
        width=output_width,
        height=output_height,
        resampleAlg="near",
        # GRA_Bilinear TODO: bilinear warp does not work with Nodata params,
        # there will be values between -999 and the valid range
        srcNodata=float(-999.0),
        dstNodata=-999.0,
    )

    # Close the files
    dataset = None
    output_dataset = None

    if output_folder is None:
        # Delete original file, rename temp file.
        os.remove(file_path)
        os.rename(output_file, file_path)

    return True

source_folder = 'data/LPDR/2017tifprueba'
output_folder = 'data/LPDR/2017tifpruebarep'

print("START")

# NOTE: Do not reproject Quality Flag files for now,
# those files end in '\d{3}QA.tif'
# Select only ascending and descending passes:
regex = r"\d{7}[AD].tif"

for file_name in os.listdir(source_folder):
    if re.findall(regex, file_name):

        file_path = os.path.join(source_folder, file_name)
        print(f"Reprojecting {file_path}")

        outcome = reproject_file(file_path, output_folder=output_folder)

        if outcome:
            print("Reprojected")
            if output_folder is not None:
                print(
                    f"""New file saved {
                        os.path.join(output_folder, file_name)
                    }"""
                )
        else:
            print(f"File {file_name} is already reprojected")

print("DONE")

START
Reprojecting data/LPDR/2017tifprueba\AMSRU_Mland_2017001A.tif


c:\Users\marce\miniconda3\envs\TFM_GDAL\lib\site-packages\osgeo\gdal.py:606: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


Reprojected
New file saved data/LPDR/2017tifpruebarep\AMSRU_Mland_2017001A.tif
Reprojecting data/LPDR/2017tifprueba\AMSRU_Mland_2017001D.tif
Reprojected
New file saved data/LPDR/2017tifpruebarep\AMSRU_Mland_2017001D.tif
DONE


## Data mixing

Since all three datasets now follow the same grid, we can go ahead and combine all three of them into a single training dataset.

### File pairing

We'll start by pairing the LPDR dataset with the correpsonding Ku and Ka datasets.

In [ ]:
def get_day_of_year(file_path):
    if '2017tif' in file_path:
        year_doyy_str = file_path.split('/')[-1].split('_')[2][:7]
        year_doyy = year_doyy_str[-3:]
    else:
        date_str = file_path.split('/')[-1].split('_')[1]
        if date_str.endswith("00"):
            return None
        datetime_obj = datetime.strptime(date_str, '%Y%m%d')
        year_doyy = datetime_obj.strftime('%Y%j')[4:]
    
    return year_doyy

def get_orientation(file_path): 
    if '2017tif' in file_path:
        orientation_str = file_path.split('/')[-1].split('_')[2][:8]
        orientation = orientation_str[-1:]
    else:
        orientation_str = file_path.split('/')[-1].split('_')[3]
        orientation = orientation_str[-1:]

    return orientation

def find_corresponding_lpdr_files(lpdr_dir, ku_dir, ka_dir):
    lpdr_files = []

    for lpdr_file in os.listdir(lpdr_dir):
        if lpdr_file.endswith(".tif") and '_QA' not in lpdr_file:
            lpdr_file_path = os.path.join(lpdr_dir, lpdr_file)
            lpdr_doyy = get_day_of_year(lpdr_file_path)
            if lpdr_doyy is None:
                continue
            lpdr_orientation = get_orientation(lpdr_file_path)

            for ku_file in os.listdir(ku_dir):
                if ku_file.endswith(".h5"):
                    ku_file_path = os.path.join(ku_dir, ku_file)
                    ku_doyy = get_day_of_year(ku_file_path)
                    if ku_doyy is None:
                        continue
                    ku_orientation = get_orientation(ku_file_path)

                    if lpdr_doyy == ku_doyy and ku_orientation == lpdr_orientation:
                        for ka_file in os.listdir(ka_dir):
                            if ka_file.endswith(".h5"):
                                ka_file_path = os.path.join(ka_dir, ka_file)
                                ka_doyy = get_day_of_year(ka_file_path)
                                if ka_doyy is None:
                                    continue
                                ka_orientation = get_orientation(ka_file_path)

                                if (lpdr_doyy == ka_doyy == ku_doyy) and \
                                (lpdr_orientation == ka_orientation == ku_orientation):
                                    lpdr_files.append((lpdr_file_path, ku_file_path, ka_file_path, lpdr_doyy, lpdr_orientation))

    return lpdr_files

lpdr_dir = "data/LPDR/2017tif/"
ku_dir = "data/18ghz/"
ka_dir = "data/36ghz/"

corresponding_files = find_corresponding_lpdr_files(lpdr_dir, ku_dir, ka_dir)

[('data/LPDR/2017tif/AMSRU_Mland_2017001A.tif',
  'data/18ghz/GW1AM2_20170101_01D_EQMA_L3SGT18LA2220220_corrected.h5',
  'data/36ghz/GW1AM2_20170101_01D_EQMA_L3SGT36LA2220220_corrected.h5',
  '001',
  'A'),
 ('data/LPDR/2017tif/AMSRU_Mland_2017001D.tif',
  'data/18ghz/GW1AM2_20170101_01D_EQMD_L3SGT18LA2220220_corrected.h5',
  'data/36ghz/GW1AM2_20170101_01D_EQMD_L3SGT36LA2220220_corrected.h5',
  '001',
  'D'),
 ('data/LPDR/2017tif/AMSRU_Mland_2017002A.tif',
  'data/18ghz/GW1AM2_20170102_01D_EQMA_L3SGT18LA2220220_corrected.h5',
  'data/36ghz/GW1AM2_20170102_01D_EQMA_L3SGT36LA2220220_corrected.h5',
  '002',
  'A'),
 ('data/LPDR/2017tif/AMSRU_Mland_2017002D.tif',
  'data/18ghz/GW1AM2_20170102_01D_EQMD_L3SGT18LA2220220_corrected.h5',
  'data/36ghz/GW1AM2_20170102_01D_EQMD_L3SGT36LA2220220_corrected.h5',
  '002',
  'D'),
 ('data/LPDR/2017tif/AMSRU_Mland_2017003A.tif',
  'data/18ghz/GW1AM2_20170103_01D_EQMA_L3SGT18LA2220220_corrected.h5',
  'data/36ghz/GW1AM2_20170103_01D_EQMA_L3SGT36LA22202

### Combining datasets

Now that we have a comprehensive list of all the files that we want to combine, we can start the combining process.

In [ ]:
def scale_factor_correction(input_file):
    corrected_ds = xr.Dataset()
    input_ds = xr.open_dataset(input_file, engine='netcdf4')

    for var in input_ds.data_vars:
        data_array = input_ds[var]
        if 'SCALE FACTOR' in data_array.attrs:
            scale_factor = data_array.attrs['SCALE FACTOR']
            corrected_ds[var] = data_array * scale_factor
        else:
            corrected_ds[var] = data_array

    return corrected_ds

def process_ldpr_dataset(lpdr_file):
    lpdr_ds = xr.open_dataset(lpdr_file, engine='netcdf4')
    
    df = lpdr_ds.to_dataframe()
    df = df.reset_index()
    
    df_pivot = df.pivot(index=['lat', 'lon', 'landmask'], columns='band', values='band_data')  
    
    df_pivot.columns = ['fwns', 'fw', 'air temperature', 'PWV', 'VOD', 'vsm', 'VPD', 'lat_rep', 'lon_rep']
    df_pivot.dropna(inplace=True)
    
    ds = df_pivot.reset_index().set_index(['lat', 'lon']).to_xarray()
    
    return ds

def process_satellite_data(lpdr_file, ka_file, ku_file):
    lpdr_ds = process_ldpr_dataset(lpdr_file)
    
    ka_ds = scale_factor_correction(ka_file)
    ku_ds = scale_factor_correction(ku_file)
    
    ka_ds = ka_ds.rename({'Brightness Temperature (H)': 'Brightness Temperature (H)_ka',
                          'Brightness Temperature (V)': 'Brightness Temperature (V)_ka'})
    
    ku_ds = ku_ds.rename({'Brightness Temperature (H)': 'Brightness Temperature (H)_ku',
                          'Brightness Temperature (V)': 'Brightness Temperature (V)_ku'})
    
    
    land_mask_transformed = lpdr_ds.copy()
    land_mask_transformed = land_mask_transformed.assign_coords({'lat': ku_ds['lat'].values, 'lon': ku_ds['lon'].values})
    lpdr_ds_final = land_mask_transformed.rename({"lon": "lon", "lat": "lat"})
    
    lpdr_ds_final = lpdr_ds_final.assign(lat_var=lpdr_ds_final['lat'], lon_var=lpdr_ds_final['lon'])
    merged_ds = xr.merge([lpdr_ds_final, ku_ds, ka_ds], compat='override')
    
    return merged_ds

def process_final_dataset_to_dataframe(final_ds):
    df = final_ds.to_dataframe()
    
    df.columns = df.columns.str.replace(' ', '_')
    df.columns = df.columns.str.replace('(', '_')
    df.columns = df.columns.str.replace(')', '')
    
    df.replace(-999.0, np.nan, inplace=True)
    df.dropna(inplace=True)
    
    df = df[df['landmask'] == 1]
    df.drop(columns='landmask', inplace=True)
    
    df.reindex()
    
    return df

output_dir = "data/Mix/"
zip_file_path = os.path.join(output_dir, "processed_data.zip")

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

with zipfile.ZipFile(zip_file_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for lpdr_file, ku_file, ka_file, lpdr_doyy, lpdr_orientation in corresponding_files:
        
        print('lpdr_file', lpdr_file)
        print('ku_file', ku_file)
        print('ka_file', ka_file)
        final_dataset = process_satellite_data(lpdr_file, ka_file, ku_file)
        processed_dataframe = process_final_dataset_to_dataframe(final_dataset)
        
        csv_buffer = io.StringIO()
        processed_dataframe.to_csv(csv_buffer, index=False)
        csv_filename = f'processed_data_{lpdr_doyy}_{lpdr_orientation}.csv'
        zipf.writestr(csv_filename, csv_buffer.getvalue())

print(f'Processed {len(corresponding_files)} files and saved to {zip_file_path}')